# One Image CNN Flow

This notebook uses **one waste image** to show how a small CNN processes an image step by step.

Today we are not training a good model. We are learning the flow:

```text
image file -> PIL image -> tensor -> convolution -> ReLU -> pooling -> flatten -> classifier output
```

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

torch.set_grad_enabled(False)
print('PyTorch:', torch.__version__)

## 2. Pick One Image

The notebook searches for the first JPG image inside `data/train`.

You can also manually set `IMAGE_PATH` to any image path.

In [ ]:
image_paths = sorted(Path('data/train').glob('*/*.jpg')) + sorted(Path('data/train').glob('*/*.jpeg'))

if not image_paths:
    raise FileNotFoundError(
        'No JPG images found in data/train. Add or unzip images first.'
    )

IMAGE_PATH = image_paths[0]
true_class = IMAGE_PATH.parent.name

print('Image path:', IMAGE_PATH)
print('Folder/class label:', true_class)

## 3. Display The Original Image

In [ ]:
image = Image.open(IMAGE_PATH).convert('RGB')
print('Original PIL image size:', image.size)

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.title(f'Original image: {true_class}')
plt.axis('off');

## 4. Convert Image To Tensor

CNNs do not read JPG files directly.

They receive a tensor shaped like:

```text
[batch_size, channels, height, width]
```

For one RGB image resized to 128x128, the shape becomes:

```text
[1, 3, 128, 128]
```

In [ ]:
image_size = 128

to_tensor = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

x = to_tensor(image)
print('Tensor shape without batch:', x.shape)

x = x.unsqueeze(0)
print('Tensor shape with batch:', x.shape)

## 5. Define A Tiny CNN

This is a small teaching CNN, not our final research model.

It has:

- `conv1`: learns 8 simple feature maps
- `conv2`: learns 16 deeper feature maps
- `pool`: reduces image size
- `fc`: converts features into 6 class scores

In [ ]:
class TinyWasteCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.fc = nn.Linear(16 * 32 * 32, num_classes)

    def forward(self, x):
        shapes = {}

        shapes['input'] = x.shape

        x = self.conv1(x)
        shapes['after conv1'] = x.shape

        x = F.relu(x)
        shapes['after relu1'] = x.shape

        x = self.pool(x)
        shapes['after pool1'] = x.shape

        x = self.conv2(x)
        shapes['after conv2'] = x.shape

        x = F.relu(x)
        shapes['after relu2'] = x.shape

        x = self.pool(x)
        shapes['after pool2'] = x.shape

        feature_maps = x.clone()

        x = torch.flatten(x, start_dim=1)
        shapes['after flatten'] = x.shape

        x = self.fc(x)
        shapes['class scores'] = x.shape

        return x, shapes, feature_maps


class_names = ['organic', 'plastic', 'paper_cardboard', 'metal', 'glass', 'other']
model = TinyWasteCNN(num_classes=len(class_names))
model.eval()

print(model)

## 6. Forward Pass

A forward pass means the image moves through the CNN once.

Because the model is not trained, the prediction is not meaningful yet. We are only studying the flow.

In [ ]:
scores, shapes, feature_maps = model(x)

for name, shape in shapes.items():
    print(f'{name:18s}: {list(shape)}')

## 7. Convert Scores To Probabilities

The final layer returns raw scores called **logits**.

`softmax` converts those scores into probabilities.

In [ ]:
probabilities = torch.softmax(scores, dim=1)[0]
predicted_index = probabilities.argmax().item()

print('True folder label:', true_class)
print('Predicted class from untrained model:', class_names[predicted_index])
print('\nProbabilities:')
for name, probability in zip(class_names, probabilities):
    print(f'{name:16s}: {probability.item():.4f}')

## 8. View Feature Maps

Feature maps are the CNN's internal visual responses.

At this stage the CNN is untrained, so these maps are not smart yet. After training, feature maps become more useful.

In [ ]:
maps = feature_maps[0].detach()
num_maps_to_show = min(8, maps.shape[0])

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
axes = axes.ravel()

for i in range(num_maps_to_show):
    axes[i].imshow(maps[i], cmap='viridis')
    axes[i].set_title(f'Feature map {i}')
    axes[i].axis('off')

for i in range(num_maps_to_show, len(axes)):
    axes[i].axis('off')

plt.tight_layout();

## 9. What To Understand Today

The most important ideas are:

- An image becomes numbers.
- CNN input shape is `[batch, channels, height, width]`.
- Convolution changes the number of channels.
- Pooling reduces height and width.
- Flatten converts image-like features into a vector.
- The final layer produces one score per class.

This is the same basic flow used inside larger CNNs like ResNet18.